In [3]:
!pip install axelrod

In [4]:
"""
IPD Strategy — Option A: History-Augmented Dueling Double DQN
=============================================================
State  : 20-dim history features
Network: Dueling DQN (value + advantage streams)
Training: Double DQN target + experience replay + curriculum (easy->hard)
Key idea: Structured state rep + dueling heads let the agent learn a strong
          baseline value V(s) independent of which action it takes — critical
          when cooperation is the dominant strategy vs most opponents.
"""

import os, random, time, json, warnings, collections
from typing import List, Dict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings("ignore")
import axelrod as axl

SEED=42; C,D=0,1; ROUNDS=100
PAYOFF={(C,C):3,(C,D):0,(D,C):5,(D,D):1}
FEAT_DIM=20
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seeds()
print(f"[Option-A: Dueling DDQN] Device: {device}")

def a2i(a): return 0 if a==axl.Action.C else 1
def i2a(i): return axl.Action.C if i==0 else axl.Action.D

def extract_features(ph, oh, rnd):
    n=max(1,rnd)
    oa=np.array(oh,dtype=np.float32) if oh else np.zeros(1,dtype=np.float32)
    pa=np.array(ph,dtype=np.float32) if ph else np.zeros(1,dtype=np.float32)
    opp_dr=float(oa.mean()); my_dr=float(pa.mean())
    r0=float(len(oh)>0 and oh[0]==D)
    r1=float(len(oh)>1 and oh[1]==D)
    r2=float(len(oh)>2 and oh[2]==D)
    def cr(ma,no):
        p=[oh[i+1] for i in range(len(ph)-1) if ph[i]==ma and i+1<len(oh)]
        return float(np.mean([b==no for b in p])) if p else 0.5
    cac=cr(C,C); dac=cr(C,D); cad=cr(D,C); dad=cr(D,D)
    mir=float(np.mean([oh[t]==ph[t-1] for t in range(1,min(len(oh),len(ph)))])) if len(ph)>=2 else 0.5
    edef=any(a==D for a in oh)
    pret=0.0
    if edef:
        fd=next(i for i,a in enumerate(oh) if a==D)
        pret=float(all(a==D for a in oh[fd:]))
    alt=float(np.mean([oh[i]!=oh[i+1] for i in range(len(oh)-1)])) if len(oh)>=2 else 0.0
    ovar=float(np.var(1-oa)) if len(oh)>=3 else 0.25
    ds=0
    for a in reversed(oh):
        if a==D: ds+=1
        else: break
    early=float(np.mean(oh[:20])) if oh else 0.5
    late=float(np.mean(oh[20:])) if len(oh)>20 else (float(np.mean(oh)) if oh else 0.5)
    mc=sum(1 for a,b in zip(ph,oh) if a==C and b==C)/n
    ex=sum(1 for a,b in zip(ph,oh) if a==D and b==C)/n
    return np.array([opp_dr,my_dr,r0,r1,r2,cac,dac,cad,dad,mir,pret,alt,ovar,
                     min(ds/10.,1.),early,late,mc,ex,rnd/ROUNDS,min(rnd/20.,1.)],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(self, in_dim=FEAT_DIM, n_actions=2, hidden=128):
        super().__init__()
        self.shared=nn.Sequential(
            nn.Linear(in_dim,hidden),nn.LayerNorm(hidden),nn.ReLU(),
            nn.Linear(hidden,hidden),nn.LayerNorm(hidden),nn.ReLU())
        self.val=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Linear(64,1))
        self.adv=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Linear(64,n_actions))
    def forward(self,x):
        h=self.shared(x); v=self.val(h); a=self.adv(h)
        return v+a-a.mean(dim=-1,keepdim=True)

Tr=collections.namedtuple("Tr",["s","a","r","s2","done"])
class ReplayBuffer:
    def __init__(self,cap=60_000): self.buf=collections.deque(maxlen=cap)
    def push(self,*args): self.buf.append(Tr(*args))
    def sample(self,n): return random.sample(self.buf,min(n,len(self.buf)))
    def __len__(self): return len(self.buf)

class IPDEnv:
    def __init__(self,opp_class,opp_kwargs=None):
        self.opp_class=opp_class; self.opp_kwargs=opp_kwargs or {}; self.reset()
    def reset(self):
        self.opp=self.opp_class(**self.opp_kwargs); self.opp.reset()
        self.ph=[]; self.oh=[]; self.rnd=0
        return extract_features([],[],0)
    def step(self,action):
        ph_ax=[i2a(a) for a in self.ph]
        class Px:
            history=ph_ax; match_winner=None; match_attributes={}
        try: oa=a2i(self.opp.strategy(Px()))
        except: oa=self.oh[-1] if self.oh else C
        rew=float(PAYOFF[(action,oa)])
        self.ph.append(action); self.oh.append(oa)
        if hasattr(self.opp,'history'):
            try: self.opp.history.append(i2a(oa),i2a(action))
            except: pass
        self.rnd+=1
        return extract_features(self.ph,self.oh,self.rnd),rew,self.rnd>=ROUNDS,{"opp_action":oa}

class DQNAgent:
    name="DQN-A"
    def __init__(self,lr=1e-3,gamma=0.95,eps_start=1.0,eps_end=0.05,
                 eps_decay=8000,batch=256,tgt_update=200):
        self.gamma=gamma; self.eps_start=eps_start; self.eps_end=eps_end
        self.eps_decay=eps_decay; self.batch=batch; self.tgt_update=tgt_update; self.steps=0
        self.qnet=DuelingDQN().to(device); self.tnet=DuelingDQN().to(device)
        self.tnet.load_state_dict(self.qnet.state_dict()); self.tnet.eval()
        self.opt=optim.Adam(self.qnet.parameters(),lr=lr)
        self.buf=ReplayBuffer(); self._ph=[]; self._oh=[]
    def eps(self): return self.eps_end+(self.eps_start-self.eps_end)*np.exp(-self.steps/self.eps_decay)
    def on_episode_start(self): self._ph=[]; self._oh=[]
    def on_episode_end(self): pass
    def on_step(self,a,o,**kw): self._ph.append(a); self._oh.append(o)
    def select_action(self,obs,eval_mode=False):
        if not eval_mode and random.random()<self.eps(): return random.randint(0,1)
        with torch.no_grad():
            return int(self.qnet(torch.tensor(obs,dtype=torch.float32).unsqueeze(0).to(device)).argmax(1).item())
    def push(self,s,a,r,s2,done): self.buf.push(s,a,r,s2,done)
    def learn(self):
        if len(self.buf)<self.batch: return 0.0
        b=self.buf.sample(self.batch)
        S=torch.tensor(np.array([t.s for t in b]),dtype=torch.float32).to(device)
        A=torch.tensor([t.a for t in b],dtype=torch.long).to(device)
        R=torch.tensor([t.r for t in b],dtype=torch.float32).to(device)
        S2=torch.tensor(np.array([t.s2 for t in b]),dtype=torch.float32).to(device)
        DN=torch.tensor([t.done for t in b],dtype=torch.float32).to(device)
        with torch.no_grad():
            na=self.qnet(S2).argmax(1)
            nq=self.tnet(S2).gather(1,na.unsqueeze(1)).squeeze(1)
            tgt=R+self.gamma*nq*(1-DN)
        cur=self.qnet(S).gather(1,A.unsqueeze(1)).squeeze(1)
        loss=F.smooth_l1_loss(cur,tgt)
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.qnet.parameters(),1.0)
        self.opt.step(); self.steps+=1
        if self.steps%self.tgt_update==0: self.tnet.load_state_dict(self.qnet.state_dict())
        return float(loss.item())

def build_pools():
    tr=[(axl.Cooperator,{},"Cooperative"),(axl.Defector,{},"Defector-Like"),
        (axl.TitForTat,{},"TFT-Like"),(axl.Grudger,{},"Grudger-Like"),
        (axl.Alternator,{},"PhaseSwitching"),(axl.TitFor2Tats,{},"TFT-Like"),
        (axl.SuspiciousTitForTat,{},"TFT-Like"),(axl.HardTitForTat,{},"TFT-Like"),
        (axl.GTFT,{},"TFT-Like"),(axl.SoftGrudger,{},"Grudger-Like"),
        (axl.Gradual,{},"Grudger-Like"),(axl.ForgetfulGrudger,{},"Grudger-Like"),
        (axl.Random,{"p":0.5},"Stochastic"),(axl.StochasticWSLS,{"ep":0.1},"Stochastic"),
        (axl.CyclerCCD,{},"PhaseSwitching"),(axl.CyclerDC,{},"PhaseSwitching"),
        (axl.Prober,{},"Defector-Like"),(axl.HardProber,{},"Defector-Like"),
        (axl.TrickyDefector,{},"Defector-Like"),(axl.Aggravater,{},"Defector-Like"),
        (axl.CyclerCCCCCD,{},"PhaseSwitching"),(axl.AdaptiveTitForTat,{},"Adaptive"),
        (axl.SlowTitForTwoTats2,{},"TFT-Like"),(axl.Random,{"p":0.3},"Stochastic"),
        (axl.GTFT,{"p":0.33},"Stochastic")]
    te=[(axl.OmegaTFT,{},"TFT-Like"),(axl.ContriteTitForTat,{},"TFT-Like"),
        (axl.EvolvedLookerUp2_2_2,{},"Grudger-Like"),
        (axl.StochasticWSLS,{"ep":0.2},"Stochastic"),
        (axl.HardProber,{},"Defector-Like"),(axl.Prober2,{},"Defector-Like"),
        (axl.ForgetfulGrudger,{},"Grudger-Like"),
        (axl.AdaptiveTitForTat,{},"Adaptive"),(axl.MetaWinner,{},"Sinusoidal"),
        (axl.Alternator,{},"PhaseSwitching"),(axl.CyclerCCCCCD,{},"PhaseSwitching")]
    mk=lambda sp:[{"class":c,"kwargs":k,"family":f,"name":str(c(**k))} for c,k,f in sp]
    return mk(tr),mk(te)

def train_agent(agent,pool,total=15_000):
    print(f"[Option-A] Training {total} episodes with curriculum...")
    t0=time.time()
    weighted=[]
    for e in pool:
        w=3 if e["family"] in ("Defector-Like","PhaseSwitching","Stochastic") else 2
        weighted.extend([e]*w)
    rews,losses=[],[]
    for ep in range(total):
        e=random.choice(weighted)
        env=IPDEnv(e["class"],e["kwargs"]); obs=env.reset()
        agent.on_episode_start(); ep_r=0.0
        for _ in range(ROUNDS):
            a=agent.select_action(obs); obs2,rew,done,info=env.step(a)
            agent.push(obs,a,rew,obs2,done); agent.on_step(a,info["opp_action"])
            l=agent.learn()
            if l: losses.append(l)
            obs=obs2; ep_r+=rew
            if done: break
        agent.on_episode_end(); rews.append(ep_r)
        if (ep+1)%3000==0:
            print(f"  Ep {ep+1:>6} | AvgR {np.mean(rews[-1000:]):>6.1f}"
                  f" | Loss {np.mean(losses[-500:]) if losses else 0:.4f}"
                  f" | eps {agent.eps():.3f} | {(time.time()-t0)/60:.1f}m")
    print(f"[Option-A] Done in {(time.time()-t0)/60:.1f}m")

def policy_tft(ph,oh,r): return C if not oh else oh[-1]
def policy_grim(ph,oh,r): return D if any(a==D for a in oh) else C
def policy_gtft(ph,oh,r):
    if not oh: return C
    return (D if random.random()>0.15 else C) if oh[-1]==D else C
def policy_tft2(ph,oh,r): return D if len(oh)>=2 and oh[-1]==D and oh[-2]==D else C

class _HC:
    def __init__(self,fn,name): self._fn=fn;self.name=name;self._ph=[];self._oh=[]
    def on_episode_start(self): self._ph=[];self._oh=[]
    def on_episode_end(self): pass
    def on_step(self,a,o,**kw): self._ph.append(a);self._oh.append(o)
    def select_action(self,obs,eval_mode=False): return self._fn(self._ph,self._oh,len(self._ph))

def make_baselines():
    return [_HC(policy_tft,"TFT"),_HC(policy_grim,"Grim"),_HC(policy_gtft,"GTFT"),
            _HC(policy_tft2,"TFT2"),_HC(lambda p,o,r:C,"AllC"),_HC(lambda p,o,r:D,"AllD")]

def evaluate(agent,pool,n_eps=200):
    res={}
    for e in pool:
        env=IPDEnv(e["class"],e["kwargs"]); oname=f"{e['family']}/{e['name']}"; rws=[]
        for _ in range(n_eps):
            agent.on_episode_start(); obs=env.reset(); ep_r=0.0
            for _ in range(ROUNDS):
                a=agent.select_action(obs,eval_mode=True); obs,rew,done,info=env.step(a)
                agent.on_step(a,info["opp_action"]); ep_r+=rew
                if done: break
            agent.on_episode_end(); rws.append(ep_r)
        res[oname]={"avg":float(np.mean(rws)),"std":float(np.std(rws)),"family":e["family"]}
    return res

def print_table(all_res,pool,title="OPTION A — Dueling DDQN"):
    names=list(all_res.keys()); onames=list(next(iter(all_res.values())).keys()); W=9
    hdr=f"{'Opponent':<34}"+"".join(f"{n:>{W}}" for n in names)
    print(f"\n{'='*len(hdr)}\n{title}\n{'='*len(hdr)}\n{hdr}\n{'-'*len(hdr)}")
    fs={n:{} for n in names}
    for o in onames:
        sh=o.split("/")[-1][:32]; vs={n:all_res[n][o]["avg"] for n in names}
        best=max(vs.values()); fam=all_res[names[0]][o]["family"]
        row=f"{sh:<34}"
        for n in names:
            mk="*" if vs[n]==best else " "; row+=f"{vs[n]:>{W-1}.1f}{mk}"
            fs[n].setdefault(fam,[]).append(vs[n])
        print(row)
    print(f"{'-'*len(hdr)}")
    row=f"{'MEAN':<34}"
    for n in names: row+=f"{np.mean([all_res[n][o]['avg'] for o in onames]):>{W}.1f}"
    print(row)
    for fam in sorted({e["family"] for e in pool}):
        row=f"  {fam:<32}"
        for n in names: row+=f"{np.mean(fs[n].get(fam,[0])):>{W}.1f}"
        print(row)
    print(f"{'='*len(hdr)}\n* = best for that opponent")
    return {n: float(np.mean([all_res[n][o]["avg"] for o in onames])) for n in names}

if __name__=="__main__":
    t0=time.time(); os.makedirs("./results",exist_ok=True)
    train_pool,test_pool=build_pools()
    agent=DQNAgent(lr=1e-3,gamma=0.95,eps_start=1.0,eps_end=0.05,
                   eps_decay=8000,batch=256,tgt_update=200)
    train_agent(agent,train_pool,total=15_000)
    print("\n[Option-A] Evaluating all agents...")
    all_res={"DQN-A":evaluate(agent,test_pool)}
    for bl in make_baselines(): all_res[bl.name]=evaluate(bl,test_pool)
    means=print_table(all_res,test_pool)
    with open("./results/option_A.json","w") as f:
        json.dump({"option":"A","means":means,"results":{a:{o:{"avg":v["avg"],"family":v["family"]}
                   for o,v in r.items()} for a,r in all_res.items()}},f,indent=2)
    print(f"\n[Option-A] Total: {(time.time()-t0)/60:.1f}m | Saved ./results/option_A.json")

[Option-A: Dueling DDQN] Device: cuda
[Option-A] Training 15000 episodes with curriculum...
  Ep   3000 | AvgR  339.6 | Loss 0.5523 | eps 0.050 | 29.9m
  Ep   6000 | AvgR  356.4 | Loss 0.4866 | eps 0.050 | 60.3m
  Ep   9000 | AvgR  356.3 | Loss 0.4819 | eps 0.050 | 90.9m
  Ep  12000 | AvgR  344.1 | Loss 0.5397 | eps 0.050 | 120.9m
  Ep  15000 | AvgR  353.4 | Loss 0.4636 | eps 0.050 | 151.1m
[Option-A] Done in 151.1m

[Option-A] Evaluating all agents...

OPTION A — Dueling DDQN
Opponent                              DQN-A      TFT     Grim     GTFT     TFT2     AllC     AllD
-------------------------------------------------------------------------------------------------
Omega TFT: 3, 8                      297.0    300.0*   300.0*   300.0*   300.0*   300.0*   104.0 
Contrite Tit For Tat                 297.0    300.0*   300.0*   300.0*   300.0*   300.0*   104.0 
EvolvedLookerUp2_2_2                  12.0    300.0*   300.0*   300.0*   300.0*   300.0*   108.0 
Stochastic WSLS: 0.2        

In [5]:
'''
IPD Strategy — Option B: DQN + Learned Opponent-Type Embedding
===============================================================
Key novelty: The Q-network receives [history_features | type_embedding].
A type classifier runs jointly with the DQN — its soft output is embedded
and concatenated to the state, so the policy conditions on opponent identity.

Joint loss: L = L_DQN + lambda * L_classification
This makes the embedding informative (not just a random projection).

Architecture:
  TypeNet  : 20 -> 64 -> 32 -> 6  (softmax)
  EmbedNet : Embedding(6, 16)   soft-weighted sum
  QNet     : (20+16) -> 128 -> 128 -> Dueling heads
'''

import os, random, time, json, warnings, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings("ignore")
import axelrod as axl

SEED=42; C,D=0,1; ROUNDS=100
PAYOFF={(C,C):3,(C,D):0,(D,C):5,(D,D):1}
FEAT_DIM=20; NUM_TYPES=6; EMB_DIM=16
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seeds()
print(f"[Option-B: Type-Conditioned DQN] Device: {device}")

def a2i(a): return 0 if a==axl.Action.C else 1
def i2a(i): return axl.Action.C if i==0 else axl.Action.D

FAMILY_TO_TYPE={"TFT-Like":0,"Grudger-Like":1,"Defector-Like":2,
                "Stochastic":3,"Cooperative":4,"PhaseSwitching":5,
                "Adaptive":0,"Sinusoidal":3}

def extract_features(ph,oh,rnd):
    n=max(1,rnd)
    oa=np.array(oh,dtype=np.float32) if oh else np.zeros(1,dtype=np.float32)
    pa=np.array(ph,dtype=np.float32) if ph else np.zeros(1,dtype=np.float32)
    opp_dr=float(oa.mean()); my_dr=float(pa.mean())
    r0=float(len(oh)>0 and oh[0]==D)
    r1=float(len(oh)>1 and oh[1]==D)
    r2=float(len(oh)>2 and oh[2]==D)
    def cr(ma,no):
        p=[oh[i+1] for i in range(len(ph)-1) if ph[i]==ma and i+1<len(oh)]
        return float(np.mean([b==no for b in p])) if p else 0.5
    cac=cr(C,C); dac=cr(C,D); cad=cr(D,C); dad=cr(D,D)
    mir=float(np.mean([oh[t]==ph[t-1] for t in range(1,min(len(oh),len(ph)))])) if len(ph)>=2 else 0.5
    edef=any(a==D for a in oh); pret=0.0
    if edef:
        fd=next(i for i,a in enumerate(oh) if a==D); pret=float(all(a==D for a in oh[fd:]))
    alt=float(np.mean([oh[i]!=oh[i+1] for i in range(len(oh)-1)])) if len(oh)>=2 else 0.0
    ovar=float(np.var(1-oa)) if len(oh)>=3 else 0.25
    ds=0
    for a in reversed(oh):
        if a==D: ds+=1
        else: break
    early=float(np.mean(oh[:20])) if oh else 0.5
    late=float(np.mean(oh[20:])) if len(oh)>20 else (float(np.mean(oh)) if oh else 0.5)
    mc=sum(1 for a,b in zip(ph,oh) if a==C and b==C)/n
    ex=sum(1 for a,b in zip(ph,oh) if a==D and b==C)/n
    return np.array([opp_dr,my_dr,r0,r1,r2,cac,dac,cad,dad,mir,pret,alt,ovar,
                     min(ds/10.,1.),early,late,mc,ex,rnd/ROUNDS,min(rnd/20.,1.)],dtype=np.float32)

class TypeConditionedDQN(nn.Module):
    def __init__(self,feat_dim=FEAT_DIM,num_types=NUM_TYPES,emb_dim=EMB_DIM,hidden=128,n_actions=2):
        super().__init__()
        self.type_net=nn.Sequential(
            nn.Linear(feat_dim,64),nn.LayerNorm(64),nn.ReLU(),
            nn.Linear(64,32),nn.ReLU(),nn.Linear(32,num_types))
        self.type_emb=nn.Embedding(num_types,emb_dim)
        aug=feat_dim+emb_dim
        self.shared=nn.Sequential(
            nn.Linear(aug,hidden),nn.LayerNorm(hidden),nn.ReLU(),
            nn.Linear(hidden,hidden),nn.LayerNorm(hidden),nn.ReLU())
        self.val=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Linear(64,1))
        self.adv=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Linear(64,n_actions))

    def get_embedding(self,x):
        logits=self.type_net(x); probs=F.softmax(logits,dim=-1)
        return probs@self.type_emb.weight, logits

    def forward(self,x):
        emb,_=self.get_embedding(x); aug=torch.cat([x,emb],dim=-1)
        h=self.shared(aug); v=self.val(h); a=self.adv(h)
        return v+a-a.mean(dim=-1,keepdim=True)

    def q_and_type(self,x):
        emb,logits=self.get_embedding(x); aug=torch.cat([x,emb],dim=-1)
        h=self.shared(aug); v=self.val(h); a=self.adv(h)
        return v+a-a.mean(dim=-1,keepdim=True), logits

Tr=collections.namedtuple("Tr",["s","a","r","s2","done","tl"])
class ReplayBuffer:
    def __init__(self,cap=60_000): self.buf=collections.deque(maxlen=cap)
    def push(self,*args): self.buf.append(Tr(*args))
    def sample(self,n): return random.sample(self.buf,min(n,len(self.buf)))
    def __len__(self): return len(self.buf)

class IPDEnv:
    def __init__(self,opp_class,opp_kwargs=None,family="unknown"):
        self.opp_class=opp_class; self.opp_kwargs=opp_kwargs or {}
        self.family=family; self.type_label=FAMILY_TO_TYPE.get(family,3); self.reset()
    def reset(self):
        self.opp=self.opp_class(**self.opp_kwargs); self.opp.reset()
        self.ph=[]; self.oh=[]; self.rnd=0; return extract_features([],[],0)
    def step(self,action):
        ph_ax=[i2a(a) for a in self.ph]
        class Px:
            history=ph_ax; match_winner=None; match_attributes={}
        try: oa=a2i(self.opp.strategy(Px()))
        except: oa=self.oh[-1] if self.oh else C
        rew=float(PAYOFF[(action,oa)]); self.ph.append(action); self.oh.append(oa)
        if hasattr(self.opp,"history"):
            try: self.opp.history.append(i2a(oa),i2a(action))
            except: pass
        self.rnd+=1
        return extract_features(self.ph,self.oh,self.rnd),rew,self.rnd>=ROUNDS,{"opp_action":oa}

class TypeConditionedAgent:
    name="DQN-B"
    def __init__(self,lr=1e-3,gamma=0.95,eps_start=1.0,eps_end=0.05,
                 eps_decay=8000,batch=256,tgt_update=200,lam_cls=0.3):
        self.gamma=gamma; self.eps_start=eps_start; self.eps_end=eps_end
        self.eps_decay=eps_decay; self.batch=batch; self.tgt_update=tgt_update
        self.lam_cls=lam_cls; self.steps=0
        self.qnet=TypeConditionedDQN().to(device); self.tnet=TypeConditionedDQN().to(device)
        self.tnet.load_state_dict(self.qnet.state_dict()); self.tnet.eval()
        self.opt=optim.Adam(self.qnet.parameters(),lr=lr,weight_decay=1e-4)
        self.buf=ReplayBuffer(); self._ph=[]; self._oh=[]
    def eps(self): return self.eps_end+(self.eps_start-self.eps_end)*np.exp(-self.steps/self.eps_decay)
    def on_episode_start(self): self._ph=[]; self._oh=[]
    def on_episode_end(self): pass
    def on_step(self,a,o,**kw): self._ph.append(a); self._oh.append(o)
    def select_action(self,obs,eval_mode=False):
        if not eval_mode and random.random()<self.eps(): return random.randint(0,1)
        with torch.no_grad():
            t=torch.tensor(obs,dtype=torch.float32).unsqueeze(0).to(device)
            return int(self.qnet(t).argmax(1).item())
    def push(self,s,a,r,s2,done,tl): self.buf.push(s,a,r,s2,done,tl)
    def learn(self):
        if len(self.buf)<self.batch: return 0.0,0.0
        b=self.buf.sample(self.batch)
        S=torch.tensor(np.array([t.s for t in b]),dtype=torch.float32).to(device)
        A=torch.tensor([t.a for t in b],dtype=torch.long).to(device)
        R=torch.tensor([t.r for t in b],dtype=torch.float32).to(device)
        S2=torch.tensor(np.array([t.s2 for t in b]),dtype=torch.float32).to(device)
        DN=torch.tensor([t.done for t in b],dtype=torch.float32).to(device)
        TL=torch.tensor([t.tl for t in b],dtype=torch.long).to(device)
        with torch.no_grad():
            na=self.qnet(S2).argmax(1)
            nq=self.tnet(S2).gather(1,na.unsqueeze(1)).squeeze(1)
            tgt=R+self.gamma*nq*(1-DN)
        q,logits=self.qnet.q_and_type(S)
        cur=q.gather(1,A.unsqueeze(1)).squeeze(1)
        ld=F.smooth_l1_loss(cur,tgt); lc=F.cross_entropy(logits,TL)
        loss=ld+self.lam_cls*lc
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.qnet.parameters(),1.0)
        self.opt.step(); self.steps+=1
        if self.steps%self.tgt_update==0: self.tnet.load_state_dict(self.qnet.state_dict())
        return float(ld.item()),float(lc.item())

def build_pools():
    tr=[(axl.Cooperator,{},"Cooperative"),(axl.Defector,{},"Defector-Like"),
        (axl.TitForTat,{},"TFT-Like"),(axl.Grudger,{},"Grudger-Like"),
        (axl.Alternator,{},"PhaseSwitching"),(axl.TitFor2Tats,{},"TFT-Like"),
        (axl.SuspiciousTitForTat,{},"TFT-Like"),(axl.HardTitForTat,{},"TFT-Like"),
        (axl.GTFT,{},"TFT-Like"),(axl.SoftGrudger,{},"Grudger-Like"),
        (axl.Gradual,{},"Grudger-Like"),(axl.ForgetfulGrudger,{},"Grudger-Like"),
        (axl.Random,{"p":0.5},"Stochastic"),(axl.StochasticWSLS,{"ep":0.1},"Stochastic"),
        (axl.CyclerCCD,{},"PhaseSwitching"),(axl.CyclerDC,{},"PhaseSwitching"),
        (axl.Prober,{},"Defector-Like"),(axl.HardProber,{},"Defector-Like"),
        (axl.TrickyDefector,{},"Defector-Like"),(axl.Aggravater,{},"Defector-Like"),
        (axl.CyclerCCCCCD,{},"PhaseSwitching"),(axl.AdaptiveTitForTat,{},"Adaptive"),
        (axl.SlowTitForTwoTats2,{},"TFT-Like"),(axl.Random,{"p":0.3},"Stochastic"),
        (axl.GTFT,{"p":0.33},"Stochastic")]
    te=[(axl.OmegaTFT,{},"TFT-Like"),(axl.ContriteTitForTat,{},"TFT-Like"),
        (axl.EvolvedLookerUp2_2_2,{},"Grudger-Like"),
        (axl.StochasticWSLS,{"ep":0.2},"Stochastic"),
        (axl.HardProber,{},"Defector-Like"),(axl.Prober2,{},"Defector-Like"),
        (axl.ForgetfulGrudger,{},"Grudger-Like"),
        (axl.AdaptiveTitForTat,{},"Adaptive"),(axl.MetaWinner,{},"Sinusoidal"),
        (axl.Alternator,{},"PhaseSwitching"),(axl.CyclerCCCCCD,{},"PhaseSwitching")]
    mk=lambda sp:[{"class":c,"kwargs":k,"family":f,"name":str(c(**k))} for c,k,f in sp]
    return mk(tr),mk(te)

def train_agent(agent,pool,total=15_000):
    print(f"[Option-B] Training {total} episodes (joint DQN+type embedding)...")
    t0=time.time()
    weighted=[]
    for e in pool:
        w=3 if e["family"] in ("Defector-Like","PhaseSwitching","Stochastic") else 2
        weighted.extend([e]*w)
    rews,dls,cls=[],[],[]
    for ep in range(total):
        e=random.choice(weighted)
        env=IPDEnv(e["class"],e["kwargs"],e["family"]); obs=env.reset()
        agent.on_episode_start(); ep_r=0.0
        for _ in range(ROUNDS):
            a=agent.select_action(obs); obs2,rew,done,info=env.step(a)
            agent.push(obs,a,rew,obs2,done,env.type_label)
            agent.on_step(a,info["opp_action"])
            ld,lc=agent.learn()
            if ld: dls.append(ld); cls.append(lc)
            obs=obs2; ep_r+=rew
            if done: break
        agent.on_episode_end(); rews.append(ep_r)
        if (ep+1)%3000==0:
            print(f"  Ep {ep+1:>6} | AvgR {np.mean(rews[-1000:]):>6.1f}"
                  f" | DQN {np.mean(dls[-500:]) if dls else 0:.4f}"
                  f" | CLS {np.mean(cls[-500:]) if cls else 0:.4f}"
                  f" | eps {agent.eps():.3f} | {(time.time()-t0)/60:.1f}m")
    print(f"[Option-B] Done in {(time.time()-t0)/60:.1f}m")

def policy_tft(ph,oh,r): return C if not oh else oh[-1]
def policy_grim(ph,oh,r): return D if any(a==D for a in oh) else C
def policy_gtft(ph,oh,r):
    if not oh: return C
    return (D if random.random()>0.15 else C) if oh[-1]==D else C
def policy_tft2(ph,oh,r): return D if len(oh)>=2 and oh[-1]==D and oh[-2]==D else C

class _HC:
    def __init__(self,fn,name): self._fn=fn;self.name=name;self._ph=[];self._oh=[]
    def on_episode_start(self): self._ph=[];self._oh=[]
    def on_episode_end(self): pass
    def on_step(self,a,o,**kw): self._ph.append(a);self._oh.append(o)
    def select_action(self,obs,eval_mode=False): return self._fn(self._ph,self._oh,len(self._ph))

def make_baselines():
    return [_HC(policy_tft,"TFT"),_HC(policy_grim,"Grim"),_HC(policy_gtft,"GTFT"),
            _HC(policy_tft2,"TFT2"),_HC(lambda p,o,r:C,"AllC"),_HC(lambda p,o,r:D,"AllD")]

def evaluate(agent,pool,n_eps=200):
    res={}
    for e in pool:
        env=IPDEnv(e["class"],e["kwargs"],e["family"])
        oname=f"{e['family']}/{e['name']}"; rws=[]
        for _ in range(n_eps):
            agent.on_episode_start(); obs=env.reset(); ep_r=0.0
            for _ in range(ROUNDS):
                a=agent.select_action(obs,eval_mode=True); obs,rew,done,info=env.step(a)
                agent.on_step(a,info["opp_action"]); ep_r+=rew
                if done: break
            agent.on_episode_end(); rws.append(ep_r)
        res[oname]={"avg":float(np.mean(rws)),"std":float(np.std(rws)),"family":e["family"]}
    return res

def print_table(all_res,pool,title="OPTION B: Type-Conditioned DQN"):
    names=list(all_res.keys()); onames=list(next(iter(all_res.values())).keys()); W=9
    hdr=f"{'Opponent':<34}"+"".join(f"{n:>{W}}" for n in names)
    print(f"\\n{'='*len(hdr)}\\n{title}\\n{'='*len(hdr)}\\n{hdr}\\n{'-'*len(hdr)}")
    fs={n:{} for n in names}
    for o in onames:
        sh=o.split("/")[-1][:32]; vs={n:all_res[n][o]["avg"] for n in names}
        best=max(vs.values()); fam=all_res[names[0]][o]["family"]
        row=f"{sh:<34}"
        for n in names:
            mk="*" if vs[n]==best else " "; row+=f"{vs[n]:>{W-1}.1f}{mk}"
            fs[n].setdefault(fam,[]).append(vs[n])
        print(row)
    print(f"{'-'*len(hdr)}")
    row=f"{'MEAN':<34}"
    for n in names: row+=f"{np.mean([all_res[n][o]['avg'] for o in onames]):>{W}.1f}"
    print(row)
    for fam in sorted({e["family"] for e in pool}):
        row=f"  {fam:<32}"
        for n in names: row+=f"{np.mean(fs[n].get(fam,[0])):>{W}.1f}"
        print(row)
    print(f"{'='*len(hdr)}\\n* = best for that opponent")
    return {n:float(np.mean([all_res[n][o]["avg"] for o in onames])) for n in names}

if __name__=="__main__":
    t0=time.time(); os.makedirs("./results",exist_ok=True)
    train_pool,test_pool=build_pools()
    agent=TypeConditionedAgent(lr=1e-3,gamma=0.95,eps_start=1.0,eps_end=0.05,
                               eps_decay=8000,batch=256,tgt_update=200,lam_cls=0.3)
    train_agent(agent,train_pool,total=15_000)
    print("\\n[Option-B] Evaluating all agents...")
    all_res={"DQN-B":evaluate(agent,test_pool)}
    for bl in make_baselines(): all_res[bl.name]=evaluate(bl,test_pool)
    means=print_table(all_res,test_pool)
    with open("./results/option_B.json","w") as f:
        json.dump({"option":"B","means":means,"results":{a:{o:{"avg":v["avg"],"family":v["family"]}
                   for o,v in r.items()} for a,r in all_res.items()}},f,indent=2)
    print(f"\\n[Option-B] Total: {(time.time()-t0)/60:.1f}m | Saved ./results/option_B.json")

[Option-B: Type-Conditioned DQN] Device: cuda
[Option-B] Training 15000 episodes (joint DQN+type embedding)...
  Ep   3000 | AvgR  330.0 | DQN 0.9584 | CLS 0.6674 | eps 0.050 | 40.3m
  Ep   6000 | AvgR  341.1 | DQN 1.0805 | CLS 0.6434 | eps 0.050 | 80.9m
  Ep   9000 | AvgR  329.6 | DQN 1.1905 | CLS 0.6380 | eps 0.050 | 121.6m
  Ep  12000 | AvgR  316.1 | DQN 1.2511 | CLS 0.5494 | eps 0.050 | 162.3m
  Ep  15000 | AvgR  294.4 | DQN 1.0839 | CLS 0.5788 | eps 0.050 | 202.8m
[Option-B] Done in 202.8m
\n[Option-B] Evaluating all agents...
\n=================================================================================================\nOPTION B: Type-Conditioned DQN\n=================================================================================================\nOpponent                              DQN-B      TFT     Grim     GTFT     TFT2     AllC     AllD\n-------------------------------------------------------------------------------------------------
Omega TFT: 3, 8                  

In [7]:
'''
IPD Strategy — Option C: Population-Based Self-Play DQN
========================================================
Key novelty: instead of training against fixed axelrod opponents only, we
maintain a POPULATION of DQN agents that co-evolve via self-play.

Training regime:
  Phase 1 (warm-up): train against axelrod pool (same as A/B)
  Phase 2 (self-play): add copies of past agent checkpoints to opponent pool
    - Every K episodes, snapshot the current agent
    - Sample opponents from {axelrod pool} U {snapshots}
    - Older snapshots sampled with exponential decay weight

Why this helps:
  - Avoids overfitting to known axelrod opponent set
  - Agent must generalize to strategies similar to itself -> promotes robustness
  - The co-evolutionary pressure pushes toward strategies that do well in
    a diverse, dynamic population — consistent with theoretical results showing
    TFT-like strategies emerge in evolutionary dynamics (Axelrod 1984, Nowak 1992)

Architecture: same Dueling DQN as Option A, but training loop differs.
This is the publishable novelty for a Q3 paper.
'''

import os, random, time, json, warnings, collections, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings("ignore")
import axelrod as axl

SEED=42; C,D=0,1; ROUNDS=100
PAYOFF={(C,C):3,(C,D):0,(D,C):5,(D,D):1}
FEAT_DIM=20
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seeds()
print(f"[Option-C: Self-Play DQN] Device: {device}")

def a2i(a): return 0 if a==axl.Action.C else 1
def i2a(i): return axl.Action.C if i==0 else axl.Action.D

def extract_features(ph,oh,rnd):
    n=max(1,rnd)
    oa=np.array(oh,dtype=np.float32) if oh else np.zeros(1,dtype=np.float32)
    pa=np.array(ph,dtype=np.float32) if ph else np.zeros(1,dtype=np.float32)
    opp_dr=float(oa.mean()); my_dr=float(pa.mean())
    r0=float(len(oh)>0 and oh[0]==D)
    r1=float(len(oh)>1 and oh[1]==D)
    r2=float(len(oh)>2 and oh[2]==D)
    def cr(ma,no):
        p=[oh[i+1] for i in range(len(ph)-1) if ph[i]==ma and i+1<len(oh)]
        return float(np.mean([b==no for b in p])) if p else 0.5
    cac=cr(C,C); dac=cr(C,D); cad=cr(D,C); dad=cr(D,D)
    mir=float(np.mean([oh[t]==ph[t-1] for t in range(1,min(len(oh),len(ph)))])) if len(ph)>=2 else 0.5
    edef=any(a==D for a in oh); pret=0.0
    if edef:
        fd=next(i for i,a in enumerate(oh) if a==D); pret=float(all(a==D for a in oh[fd:]))
    alt=float(np.mean([oh[i]!=oh[i+1] for i in range(len(oh)-1)])) if len(oh)>=2 else 0.0
    ovar=float(np.var(1-oa)) if len(oh)>=3 else 0.25
    ds=0
    for a in reversed(oh):
        if a==D: ds+=1
        else: break
    early=float(np.mean(oh[:20])) if oh else 0.5
    late=float(np.mean(oh[20:])) if len(oh)>20 else (float(np.mean(oh)) if oh else 0.5)
    mc=sum(1 for a,b in zip(ph,oh) if a==C and b==C)/n
    ex=sum(1 for a,b in zip(ph,oh) if a==D and b==C)/n
    return np.array([opp_dr,my_dr,r0,r1,r2,cac,dac,cad,dad,mir,pret,alt,ovar,
                     min(ds/10.,1.),early,late,mc,ex,rnd/ROUNDS,min(rnd/20.,1.)],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(self,in_dim=FEAT_DIM,n_actions=2,hidden=128):
        super().__init__()
        self.shared=nn.Sequential(
            nn.Linear(in_dim,hidden),nn.LayerNorm(hidden),nn.ReLU(),
            nn.Linear(hidden,hidden),nn.LayerNorm(hidden),nn.ReLU())
        self.val=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Linear(64,1))
        self.adv=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Linear(64,n_actions))
    def forward(self,x):
        h=self.shared(x); v=self.val(h); a=self.adv(h)
        return v+a-a.mean(dim=-1,keepdim=True)

Tr=collections.namedtuple("Tr",["s","a","r","s2","done"])
class ReplayBuffer:
    def __init__(self,cap=60_000): self.buf=collections.deque(maxlen=cap)
    def push(self,*args): self.buf.append(Tr(*args))
    def sample(self,n): return random.sample(self.buf,min(n,len(self.buf)))
    def __len__(self): return len(self.buf)

class IPDEnv:
    def __init__(self,opp_class,opp_kwargs=None):
        self.opp_class=opp_class; self.opp_kwargs=opp_kwargs or {}; self.reset()
    def reset(self):
        self.opp=self.opp_class(**self.opp_kwargs); self.opp.reset()
        self.ph=[]; self.oh=[]; self.rnd=0; return extract_features([],[],0)
    def step(self,action):
        ph_ax=[i2a(a) for a in self.ph]
        class Px:
            history=ph_ax; match_winner=None; match_attributes={}
        try: oa=a2i(self.opp.strategy(Px()))
        except: oa=self.oh[-1] if self.oh else C
        rew=float(PAYOFF[(action,oa)]); self.ph.append(action); self.oh.append(oa)
        if hasattr(self.opp,"history"):
            try: self.opp.history.append(i2a(oa),i2a(action))
            except: pass
        self.rnd+=1
        return extract_features(self.ph,self.oh,self.rnd),rew,self.rnd>=ROUNDS,{"opp_action":oa}

# ─── FROZEN SNAPSHOT OPPONENT ────────────────────────────────────────────────
class SnapshotOpponent:
    # \"\"\"A frozen DQN checkpoint that plays greedily as an opponent.\"\"\"
    def __init__(self,net_state_dict,eps=0.05):
        self.net=DuelingDQN().to(device)
        self.net.load_state_dict(net_state_dict)
        self.net.eval(); self.eps=eps
        self.ph=[]; self.oh=[]
    def reset(self): self.ph=[]; self.oh=[]
    def act(self,obs):
        if random.random()<self.eps: return random.randint(0,1)
        with torch.no_grad():
            t=torch.tensor(obs,dtype=torch.float32).unsqueeze(0).to(device)
            return int(self.net(t).argmax(1).item())

class SelfPlayEnv:
    # \"\"\"Episode against a SnapshotOpponent — both players see own-perspective features.\"\"\"
    def __init__(self,snapshot: SnapshotOpponent):
        self.snap=snapshot; self.reset()
    def reset(self):
        self.snap.reset()
        self.ph=[]; self.oh=[]; self.rnd=0; return extract_features([],[],0)
    def step(self,action):
        opp_obs=extract_features(self.oh,self.ph,self.rnd)  # opponent's perspective
        oa=self.snap.act(opp_obs)
        rew=float(PAYOFF[(action,oa)]); self.ph.append(action); self.oh.append(oa)
        self.rnd+=1
        return extract_features(self.ph,self.oh,self.rnd),rew,self.rnd>=ROUNDS,{"opp_action":oa}

# ─── DQN AGENT ───────────────────────────────────────────────────────────────
class SelfPlayDQNAgent:
    name="DQN-C"
    def __init__(self,lr=1e-3,gamma=0.95,eps_start=1.0,eps_end=0.05,
                 eps_decay=8000,batch=256,tgt_update=200,
                 snapshot_every=500,max_snapshots=20,selfplay_frac=0.4):
        self.gamma=gamma; self.eps_start=eps_start; self.eps_end=eps_end
        self.eps_decay=eps_decay; self.batch=batch; self.tgt_update=tgt_update
        self.snapshot_every=snapshot_every; self.max_snapshots=max_snapshots
        self.selfplay_frac=selfplay_frac; self.steps=0
        self.qnet=DuelingDQN().to(device); self.tnet=DuelingDQN().to(device)
        self.tnet.load_state_dict(self.qnet.state_dict()); self.tnet.eval()
        self.opt=optim.Adam(self.qnet.parameters(),lr=lr)
        self.buf=ReplayBuffer(); self._ph=[]; self._oh=[]
        self.snapshots=[]   # list of (state_dict, age)
    def eps(self): return self.eps_end+(self.eps_start-self.eps_end)*np.exp(-self.steps/self.eps_decay)
    def on_episode_start(self): self._ph=[]; self._oh=[]
    def on_episode_end(self): pass
    def on_step(self,a,o,**kw): self._ph.append(a); self._oh.append(o)
    def select_action(self,obs,eval_mode=False):
        if not eval_mode and random.random()<self.eps(): return random.randint(0,1)
        with torch.no_grad():
            t=torch.tensor(obs,dtype=torch.float32).unsqueeze(0).to(device)
            return int(self.qnet(t).argmax(1).item())
    def push(self,s,a,r,s2,done): self.buf.push(s,a,r,s2,done)
    def learn(self):
        if len(self.buf)<self.batch: return 0.0
        b=self.buf.sample(self.batch)
        S=torch.tensor(np.array([t.s for t in b]),dtype=torch.float32).to(device)
        A=torch.tensor([t.a for t in b],dtype=torch.long).to(device)
        R=torch.tensor([t.r for t in b],dtype=torch.float32).to(device)
        S2=torch.tensor(np.array([t.s2 for t in b]),dtype=torch.float32).to(device)
        DN=torch.tensor([t.done for t in b],dtype=torch.float32).to(device)
        with torch.no_grad():
            na=self.qnet(S2).argmax(1)
            nq=self.tnet(S2).gather(1,na.unsqueeze(1)).squeeze(1)
            tgt=R+self.gamma*nq*(1-DN)
        cur=self.qnet(S).gather(1,A.unsqueeze(1)).squeeze(1)
        loss=F.smooth_l1_loss(cur,tgt)
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.qnet.parameters(),1.0)
        self.opt.step(); self.steps+=1
        if self.steps%self.tgt_update==0: self.tnet.load_state_dict(self.qnet.state_dict())
        return float(loss.item())
    def maybe_snapshot(self,ep):
        if ep>0 and ep%self.snapshot_every==0:
            sd=copy.deepcopy(self.qnet.state_dict())
            self.snapshots.append(sd)
            if len(self.snapshots)>self.max_snapshots:
                self.snapshots.pop(0)
    def sample_snapshot(self):
        if not self.snapshots: return None
        # Exponential recency weighting: newer snapshots more likely
        n=len(self.snapshots)
        weights=np.exp(np.linspace(0,2,n)); weights/=weights.sum()
        idx=np.random.choice(n,p=weights)
        return SnapshotOpponent(self.snapshots[idx],eps=0.05)

def build_pools():
    tr=[(axl.Cooperator,{},"Cooperative"),(axl.Defector,{},"Defector-Like"),
        (axl.TitForTat,{},"TFT-Like"),(axl.Grudger,{},"Grudger-Like"),
        (axl.Alternator,{},"PhaseSwitching"),(axl.TitFor2Tats,{},"TFT-Like"),
        (axl.SuspiciousTitForTat,{},"TFT-Like"),(axl.HardTitForTat,{},"TFT-Like"),
        (axl.GTFT,{},"TFT-Like"),(axl.SoftGrudger,{},"Grudger-Like"),
        (axl.Gradual,{},"Grudger-Like"),(axl.ForgetfulGrudger,{},"Grudger-Like"),
        (axl.Random,{"p":0.5},"Stochastic"),(axl.StochasticWSLS,{"ep":0.1},"Stochastic"),
        (axl.CyclerCCD,{},"PhaseSwitching"),(axl.CyclerDC,{},"PhaseSwitching"),
        (axl.Prober,{},"Defector-Like"),(axl.HardProber,{},"Defector-Like"),
        (axl.TrickyDefector,{},"Defector-Like"),(axl.Aggravater,{},"Defector-Like"),
        (axl.CyclerCCCCCD,{},"PhaseSwitching"),(axl.AdaptiveTitForTat,{},"Adaptive"),
        (axl.SlowTitForTwoTats2,{},"TFT-Like"),(axl.Random,{"p":0.3},"Stochastic"),
        (axl.GTFT,{"p":0.33},"Stochastic")]
    te=[(axl.OmegaTFT,{},"TFT-Like"),(axl.ContriteTitForTat,{},"TFT-Like"),
        (axl.EvolvedLookerUp2_2_2,{},"Grudger-Like"),
        (axl.StochasticWSLS,{"ep":0.2},"Stochastic"),
        (axl.HardProber,{},"Defector-Like"),(axl.Prober2,{},"Defector-Like"),
        (axl.ForgetfulGrudger,{},"Grudger-Like"),
        (axl.AdaptiveTitForTat,{},"Adaptive"),(axl.MetaWinner,{},"Sinusoidal"),
        (axl.Alternator,{},"PhaseSwitching"),(axl.CyclerCCCCCD,{},"PhaseSwitching")]
    mk=lambda sp:[{"class":c,"kwargs":k,"family":f,"name":str(c(**k))} for c,k,f in sp]
    return mk(tr),mk(te)

def train_agent(agent,pool,total=15_000,warmup=5_000):
    print(f"[Option-C] Training {total} eps (warmup={warmup}, self-play from ep {warmup})...")
    t0=time.time()
    weighted=[]
    for e in pool:
        w=3 if e["family"] in ("Defector-Like","PhaseSwitching","Stochastic") else 2
        weighted.extend([e]*w)
    rews,losses=[],[]
    for ep in range(total):
        agent.maybe_snapshot(ep)
        use_selfplay=(ep>=warmup and len(agent.snapshots)>0
                      and random.random()<agent.selfplay_frac)
        if use_selfplay:
            snap=agent.sample_snapshot(); senv=SelfPlayEnv(snap)
            obs=senv.reset(); agent.on_episode_start(); ep_r=0.0
            for _ in range(ROUNDS):
                a=agent.select_action(obs); obs2,rew,done,info=senv.step(a)
                agent.push(obs,a,rew,obs2,done); agent.on_step(a,info["opp_action"])
                l=agent.learn()
                if l: losses.append(l)
                obs=obs2; ep_r+=rew
                if done: break
        else:
            e=random.choice(weighted)
            env=IPDEnv(e["class"],e["kwargs"]); obs=env.reset()
            agent.on_episode_start(); ep_r=0.0
            for _ in range(ROUNDS):
                a=agent.select_action(obs); obs2,rew,done,info=env.step(a)
                agent.push(obs,a,rew,obs2,done); agent.on_step(a,info["opp_action"])
                l=agent.learn()
                if l: losses.append(l)
                obs=obs2; ep_r+=rew
                if done: break
        agent.on_episode_end(); rews.append(ep_r)
        if (ep+1)%3000==0:
            sp_mode="(self-play active)" if ep>=warmup else "(warmup)"
            print(f"  Ep {ep+1:>6} | AvgR {np.mean(rews[-1000:]):>6.1f}"
                  f" | Loss {np.mean(losses[-500:]) if losses else 0:.4f}"
                  f" | eps {agent.eps():.3f} | snaps {len(agent.snapshots)}"
                  f" | {(time.time()-t0)/60:.1f}m {sp_mode}")
    print(f"[Option-C] Done in {(time.time()-t0)/60:.1f}m")

def policy_tft(ph,oh,r): return C if not oh else oh[-1]
def policy_grim(ph,oh,r): return D if any(a==D for a in oh) else C
def policy_gtft(ph,oh,r):
    if not oh: return C
    return (D if random.random()>0.15 else C) if oh[-1]==D else C
def policy_tft2(ph,oh,r): return D if len(oh)>=2 and oh[-1]==D and oh[-2]==D else C

class _HC:
    def __init__(self,fn,name): self._fn=fn;self.name=name;self._ph=[];self._oh=[]
    def on_episode_start(self): self._ph=[];self._oh=[]
    def on_episode_end(self): pass
    def on_step(self,a,o,**kw): self._ph.append(a);self._oh.append(o)
    def select_action(self,obs,eval_mode=False): return self._fn(self._ph,self._oh,len(self._ph))

def make_baselines():
    return [_HC(policy_tft,"TFT"),_HC(policy_grim,"Grim"),_HC(policy_gtft,"GTFT"),
            _HC(policy_tft2,"TFT2"),_HC(lambda p,o,r:C,"AllC"),_HC(lambda p,o,r:D,"AllD")]

def evaluate(agent,pool,n_eps=200):
    res={}
    for e in pool:
        env=IPDEnv(e["class"],e["kwargs"])
        oname=f"{e['family']}/{e['name']}"; rws=[]
        for _ in range(n_eps):
            agent.on_episode_start(); obs=env.reset(); ep_r=0.0
            for _ in range(ROUNDS):
                a=agent.select_action(obs,eval_mode=True); obs,rew,done,info=env.step(a)
                agent.on_step(a,info["opp_action"]); ep_r+=rew
                if done: break
            agent.on_episode_end(); rws.append(ep_r)
        res[oname]={"avg":float(np.mean(rws)),"std":float(np.std(rws)),"family":e["family"]}
    return res

def print_table(all_res,pool,title="OPTION C: Self-Play DQN"):
    names=list(all_res.keys()); onames=list(next(iter(all_res.values())).keys()); W=9
    hdr=f"{'Opponent':<34}"+"".join(f"{n:>{W}}" for n in names)
    print(f"\\n{'='*len(hdr)}\\n{title}\\n{'='*len(hdr)}\\n{hdr}\\n{'-'*len(hdr)}")
    fs={n:{} for n in names}
    for o in onames:
        sh=o.split("/")[-1][:32]; vs={n:all_res[n][o]["avg"] for n in names}
        best=max(vs.values()); fam=all_res[names[0]][o]["family"]
        row=f"{sh:<34}"
        for n in names:
            mk="*" if vs[n]==best else " "; row+=f"{vs[n]:>{W-1}.1f}{mk}"
            fs[n].setdefault(fam,[]).append(vs[n])
        print(row)
    print(f"{'-'*len(hdr)}")
    row=f"{'MEAN':<34}"
    for n in names: row+=f"{np.mean([all_res[n][o]['avg'] for o in onames]):>{W}.1f}"
    print(row)
    for fam in sorted({e["family"] for e in pool}):
        row=f"  {fam:<32}"
        for n in names: row+=f"{np.mean(fs[n].get(fam,[0])):>{W}.1f}"
        print(row)
    print(f"{'='*len(hdr)}\\n* = best for that opponent")
    return {n:float(np.mean([all_res[n][o]["avg"] for o in onames])) for n in names}

if __name__=="__main__":
    t0=time.time(); os.makedirs("./results",exist_ok=True)
    train_pool,test_pool=build_pools()
    agent=SelfPlayDQNAgent(lr=1e-3,gamma=0.95,eps_start=1.0,eps_end=0.05,
                           eps_decay=8000,batch=256,tgt_update=200,
                           snapshot_every=500,max_snapshots=20,selfplay_frac=0.4)
    train_agent(agent,train_pool,total=15_000,warmup=5_000)
    print("\\n[Option-C] Evaluating all agents...")
    all_res={"DQN-C":evaluate(agent,test_pool)}
    for bl in make_baselines(): all_res[bl.name]=evaluate(bl,test_pool)
    means=print_table(all_res,test_pool)
    with open("./results/option_C.json","w") as f:
        json.dump({"option":"C","means":means,"results":{a:{o:{"avg":v["avg"],"family":v["family"]}
                   for o,v in r.items()} for a,r in all_res.items()}},f,indent=2)
    print(f"\\n[Option-C] Total: {(time.time()-t0)/60:.1f}m | Saved ./results/option_C.json")


[Option-C: Self-Play DQN] Device: cuda
[Option-C] Training 15000 eps (warmup=5000, self-play from ep 5000)...
  Ep   3000 | AvgR  339.6 | Loss 0.5523 | eps 0.050 | snaps 5 | 29.3m (warmup)
  Ep   6000 | AvgR  320.7 | Loss 1.5322 | eps 0.050 | snaps 11 | 60.3m (self-play active)
  Ep   9000 | AvgR  304.2 | Loss 1.5824 | eps 0.050 | snaps 17 | 92.1m (self-play active)
  Ep  12000 | AvgR  285.3 | Loss 2.0828 | eps 0.050 | snaps 20 | 124.2m (self-play active)
  Ep  15000 | AvgR  297.0 | Loss 1.3412 | eps 0.050 | snaps 20 | 155.9m (self-play active)
[Option-C] Done in 155.9m
\n[Option-C] Evaluating all agents...
\n=================================================================================================\nOPTION C: Self-Play DQN\n=================================================================================================\nOpponent                              DQN-C      TFT     Grim     GTFT     TFT2     AllC     AllD\n-------------------------------------------------------------